# Actor-Critic and GAE — interactive companion

Companion to [Post 2b: Actor-Critic and variance reduction](../posts/02b-actor-critic.qmd).
REINFORCE plus a *learned* baseline. The actor proposes; the critic
evaluates. Together they reduce gradient variance dramatically.

**What you'll do (≈ 20 minutes):**
1. Implement GAE (generalized advantage estimation) in 8 lines.
2. Train A2C and watch the critic catch up to V*.
3. Sweep the GAE λ parameter and see the bias–variance trade-off live.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["figure.dpi"] = 110

from nano_agents.mdp import TwoGoalGridWorld, value_iteration
from nano_agents.policy_gradient import (
    SoftmaxPolicy, TabularBaseline,
    train_a2c, train_reinforce, gae,
)

## 1. GAE in 8 lines

The Generalized Advantage Estimator interpolates smoothly between
TD(0) (low variance, biased) and Monte Carlo (unbiased, high variance):

$$
A_t^{\text{GAE}(\gamma,\lambda)} \;=\; \sum_{k=0}^{\infty} (\gamma\lambda)^k\,\delta_{t+k},
\quad \delta_t = r_{t+1} + \gamma V(s_{t+1}) - V(s_t)
$$

Computed backward through the trajectory:

In [ ]:
def my_gae(rewards, values, next_value, gamma, lam):
    T = len(rewards)
    A = np.zeros(T)
    last = 0.0
    for t in reversed(range(T)):
        v_next = next_value if t == T - 1 else values[t + 1]
        delta = rewards[t] + gamma * v_next - values[t]
        A[t] = delta + gamma * lam * last
        last = A[t]
    return A

# Sanity check on a toy trajectory.
rewards = [1.0, 0.5, -0.2]
values = np.array([0.3, 0.2, 0.1])
A0 = my_gae(rewards, values, next_value=0.0, gamma=0.9, lam=0.0)  # TD(0)
A1 = my_gae(rewards, values, next_value=0.0, gamma=0.9, lam=1.0)  # MC
print(f"λ = 0.0 (TD(0)):     {A0}")
print(f"λ = 1.0 (MC - V):    {A1}")

- $\lambda = 0$: pure TD(0). Each advantage uses one real reward plus the bootstrap.
- $\lambda = 1$: Monte Carlo. Each advantage is the full discounted return minus $V$.
- Intermediate: weighted mix. PPO defaults to $\lambda = 0.95$.

### Try this
- Set $\lambda = 0.5$. Do the advantages still pass sanity checks vs the
  manual MC and TD calculations?
- What if $V$ values are very wrong (e.g., all zeros)? Compare the
  advantages at $\lambda = 0$ vs $\lambda = 1$ — one of them is much
  more affected.

## 2. A2C training

Same gridworld as REINFORCE. We track:
- Episode returns over time.
- $\|V_\phi - V^\star\|_\infty$ — how accurate is the critic?

In [ ]:
env = TwoGoalGridWorld(slip=0.0, step_reward=-0.04)
gamma = 0.95
V_star, _, _ = value_iteration(env, gamma=gamma)

policy = SoftmaxPolicy(env.nS, env.nA)
baseline = TabularBaseline(env.nS)

# Train A2C in chunks so we can probe the critic's accuracy.
n_chunks = 20
chunk_size = 100
critic_errors = np.zeros(n_chunks)
returns_history = []
rng = np.random.default_rng(0)
for c in range(n_chunks):
    h = train_a2c(env, policy, baseline,
                  n_episodes=chunk_size,
                  lr_actor=0.05, lr_critic=0.2,
                  gamma=gamma, lam=0.95, rng=rng)
    returns_history.extend(h["returns"])
    critic_errors[c] = np.max(np.abs(baseline.V - V_star))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
w = 30
ma = np.convolve(returns_history, np.ones(w)/w, mode="valid")
axes[0].plot(ma); axes[0].set_xlabel("episode")
axes[0].set_ylabel(f"return ({w}-ep MA)"); axes[0].set_title("A2C+GAE learning curve")
axes[0].grid(alpha=0.3)
axes[1].semilogy(np.arange(n_chunks)*chunk_size, critic_errors, "o-")
axes[1].set_xlabel("episode"); axes[1].set_ylabel(r"$\|V_\phi - V^\star\|_\infty$ (log)")
axes[1].set_title("Critic catches up to V*")
axes[1].grid(which="both", alpha=0.3)
plt.tight_layout(); plt.show()
print(f"Final critic error: {critic_errors[-1]:.3f}")

The critic error drops from ~10 to ~1 as the policy stabilizes. **Important
observation**: $V_\phi$ converges to $V^{\pi_\theta}$ (the *current policy's*
value), not $V^\star$. Once the policy stops visiting some states (e.g.,
near the small +1 goal), $V_\phi$ stops getting accurate updates there.

### Try this
- Set `lam=0.0` (pure TD). How does the critic's error trajectory change?
- Set `lr_critic=0.05` (slow critic). Does the actor still learn?
- Set `lr_critic=1.0` (aggressive critic). Stable or unstable?

## 3. The GAE λ sweep — live bias/variance trade-off

Run A2C with several λ values on a slippery gridworld. λ = 0 has the
highest bias (when the critic is wrong), λ = 1 has the highest variance.
The sweet spot is usually 0.9–0.97.

In [ ]:
from nano_agents.mdp import GridWorld

env2 = GridWorld(rows=5, cols=5,
                  terminals={(4, 4): 1.0, (0, 4): -1.0},
                  step_reward=-0.04, slip=0.2)
n_seeds = 3
n_episodes = 1500
lams = [0.0, 0.3, 0.6, 0.95, 1.0]

finals = {lam: [] for lam in lams}
for lam in lams:
    for seed in range(n_seeds):
        policy = SoftmaxPolicy(env2.nS, env2.nA)
        baseline = TabularBaseline(env2.nS)
        h = train_a2c(env2, policy, baseline, n_episodes=n_episodes,
                       lr_actor=0.05, lr_critic=0.2,
                       gamma=gamma, lam=lam,
                       rng=np.random.default_rng(seed))
        finals[lam].append(float(np.mean(h["returns"][-200:])))

means = [np.mean(finals[lam]) for lam in lams]
sems = [np.std(finals[lam]) / np.sqrt(n_seeds) for lam in lams]
plt.errorbar(lams, means, yerr=sems, fmt="o-", linewidth=2,
              markersize=10, capsize=4)
plt.xlabel(r"GAE $\lambda$"); plt.ylabel("final average return")
plt.title(f"GAE λ sweep on slippery gridworld ({n_seeds} seeds)")
plt.grid(alpha=0.3); plt.show()
print({f"λ={lam}": f"{m:.3f}" for lam, m in zip(lams, means)})

**Honest result**: on this small problem, the differences are small but
meaningful. λ = 0 (pure TD) is the worst — it relies entirely on the noisy
critic. Mid-to-high λ is the sweet spot.

The benefit of GAE scales with how unreliable the critic is. On real
problems with deep networks and noisy rewards, λ ∈ [0.9, 0.97] reliably
wins.

### Try this
- Make the problem harder (`slip=0.4`, 8×8 grid). Does the λ effect
  become more pronounced?
- Set the rewards to be sparse (terminals only, no `step_reward`).
  GAE matters more when the return signal is delayed.

## What's next

A2C is the algorithmic chassis under PPO, SAC, GRPO, and most modern
policy-gradient methods. The one big remaining problem: vanilla A2C
breaks at large learning rates (same lr-sensitivity issue as REINFORCE).

The fix is a **trust region** — limit how much the policy can change per
update. That's PPO. Open
[`02c-ppo.ipynb`](02c-ppo.ipynb).